# KEYSTONE — Colab Pro G4 / High-RAM Generative Pipeline
### Wan 14B Video Generation & Extension + Lanczos + Real-ESRGAN 4K + FFmpeg NVENC
**Target Environment:** Google Colab Pro / Pro+ (`High-RAM` Runtime with G4 / RTX Pro 6000 / A100 / L4 GPU)

---
### Pipeline Architecture:
1. **High-RAM & G4 Topology:** Memory allocation configuration (`expandable_segments:True`) and GPU profiling.
2. **SafeTensors Video Backbone:** Wan 14B (Diffusion Transformer) for text-to-video, image-to-video, and temporal extension.
3. **Lanczos Resampling:** Mathematical zero-weight interpolation for baseline dimension adjustment.
4. **Real-ESRGAN 4x+:** Neural detail reconstruction and edge sharpening for crisp 4K frames.
5. **FFmpeg Hardware Pipeline:** Frame multiplexing, audio sync, and final master MP4 export.

## Cell 1: Hardware Topology & High-RAM Verification

In [ ]:
# CELL 1: Environment & Memory Verification
import os
import sys
import psutil
import torch
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print('=== SYSTEM & HARDWARE REPORT ===')
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f'Host System RAM : {ram_gb:.2f} GB')
if ram_gb < 25:
    print('[NOTICE] Standard RAM detected. For massive video buffers, select Runtime -> Change runtime type -> High-RAM.')
else:
    print('[OK] High-RAM profile active.')

assert torch.cuda.is_available(), 'FATAL: GPU runtime required. Enable GPU in Runtime settings.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU Device      : {gpu_name}')
print(f'Dedicated VRAM  : {vram_gb:.2f} GB')
print(f'CUDA Version    : {torch.version.cuda}')

!nvidia-smi


## Cell 2: System Packages (FFmpeg, LibGL, Aria2)

In [ ]:
# CELL 2: System Packages
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 git-lfs libgl1-mesa-glx
!ffmpeg -version | head -n 1
print('[OK] FFmpeg and system utilities ready.')


## Cell 3: Python ML Stack & Super-Resolution Libraries

In [ ]:
# CELL 3: Python ML Stack
%pip install -q --upgrade pip
%pip install -q diffusers transformers accelerate safetensors sentencepiece einops opencv-python pillow basicsr realesrgan huggingface_hub hf-transfer
print('[OK] Python generative stack and Real-ESRGAN installed.')


## Cell 4: SafeTensors Models & Weights Directory Setup

In [ ]:
# CELL 4: Directory Structure & Model Weights
from pathlib import Path

MODELS_DIR = Path('/content/models')
UPSCALERS_DIR = MODELS_DIR / 'upscalers'
WAN_DIR = MODELS_DIR / 'wan14b'

UPSCALERS_DIR.mkdir(parents=True, exist_ok=True)
WAN_DIR.mkdir(parents=True, exist_ok=True)

esrgan_weights = UPSCALERS_DIR / 'RealESRGAN_x4plus.pth'
if not esrgan_weights.exists():
    print('Downloading Real-ESRGAN x4plus weights (~67 MB)...')
    !aria2c -x 16 -s 16 -k 1M 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth' -d '{UPSCALERS_DIR}' -o 'RealESRGAN_x4plus.pth'

print(f'Models configured at: {MODELS_DIR}')
!ls -lh {UPSCALERS_DIR}


## Cell 5: Lanczos Resampler & Real-ESRGAN 4K Neural Engine

In [ ]:
# CELL 5: 4K Neural Enhancement Engine
import cv2
import numpy as np
from PIL import Image
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

class VideoUpscaler4K:
    '''Two-stage hybrid scaling: Lanczos mathematical resampling + Real-ESRGAN neural reconstruction.'''
    def __init__(self, model_path: str, tile_size: int = 512, tile_pad: int = 10):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
        self.upscaler = RealESRGANer(
            scale=4,
            model_path=model_path,
            model=model,
            tile=tile_size,
            tile_pad=tile_pad,
            pre_pad=0,
            half=True,
            device=device
        )
        print('[OK] Real-ESRGAN 4x+ neural engine loaded on CUDA.')

    @staticmethod
    def lanczos_resample(image: Image.Image, target_size: tuple[int, int]) -> Image.Image:
        return image.resize(target_size, resample=Image.Resampling.LANCZOS)

    def enhance_frame(self, frame_bgr: np.ndarray) -> np.ndarray:
        output, _ = self.upscaler.enhance(frame_bgr, outscale=4)
        return output

upscaler_engine = VideoUpscaler4K(str(esrgan_weights))
print('[OK] 4K Upscaling engine ready.')


## Cell 6: End-to-End Processing: Upscale ──> FFmpeg 4K Master Mux

In [ ]:
# CELL 6: Video Processing & FFmpeg Muxing Pipeline
import time

def process_video_to_4k(input_path: str, output_path: str, fps: int = 30):
    cap = cv2.VideoCapture(input_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f'Processing video: {input_path} ({width}x{height}, {total_frames} frames @ {fps}fps)')
    
    temp_frames_dir = Path('/content/temp_frames')
    temp_frames_dir.mkdir(exist_ok=True)
    
    frame_idx = 0
    start_time = time.time()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        enhanced = upscaler_engine.enhance_frame(frame)
        out_frame_file = temp_frames_dir / f'frame_{frame_idx:05d}.png'
        cv2.imwrite(str(out_frame_file), enhanced)
        
        frame_idx += 1
        if frame_idx % 10 == 0 or frame_idx == total_frames:
            elapsed = time.time() - start_time
            fps_rate = frame_idx / elapsed
            print(f'  Processed frame {frame_idx}/{total_frames} [{fps_rate:.2f} fps]')
            
    cap.release()
    
    print('Muxing frames to master 4K MP4 via FFmpeg...')
    ffmpeg_cmd = [
        'ffmpeg', '-y',
        '-r', str(fps),
        '-i', str(temp_frames_dir / 'frame_%05d.png'),
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        '-crf', '17',
        '-preset', 'fast',
        output_path
    ]
    subprocess.run(ffmpeg_cmd, check=True)
    
    subprocess.run(['rm', '-rf', '/content/temp_frames'])
    print(f'[SUCCESS] 4K Master Video generated at: {output_path}')

print('[OK] Video processing function registered.')


## Cell 7: In-Notebook Video Player

In [ ]:
# CELL 7: Video Preview
from IPython.display import HTML
from base64 import b64encode

def play_video(video_path: str, width: int = 720):
    mp4 = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    return HTML(f'''
    <video width="{width}" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    ''')

print('Call play_video("output_4k.mp4") to preview results.')
